# ATLAS Solar Scenario Delta Correction Tutorial

This notebook combines the present ATLAS dataset with downscaled climate scenario outputs.

The workflow is designed for users who only need to edit the input parameters and run the notebook step by step.

The method is:

1. Open the present ATLAS data from `atlas_path`.
2. Open the downscaled historical simulation.
3. Open the downscaled future scenario simulation.
4. Compute the climate change delta as `future scenario minus historical simulation`.
5. Add the delta to the present ATLAS dataset.
6. Save the corrected future ATLAS dataset.


## Step 1. Import Python libraries

Run this cell first. It loads the libraries used by the workflow.


In [1]:
from pathlib import Path
import warnings

import numpy as np
import xarray as xr
from dask.diagnostics import ProgressBar

warnings.filterwarnings("ignore")


## Step 2. Set input parameters

Edit only this cell before running the notebook.

Path convention used by this notebook:

- `atlas_path`: present ATLAS data produced by the previous scaling notebook.
- `historical_downscaling_path`: downscaled historical model output.
- `scenario_downscaling_path`: downscaled future scenario model output.
- `output_path`: corrected future ATLAS data.

The output folder includes `country`, `model`, and `experiment`, so different scenario runs are kept separated.


In [2]:
# Country name used in file and folder names.
country = "argentina"

# Solar variable used in the ATLAS workflow.
variable = "ssrd"

# Downscaled model variable name used in the climate model files.
target = "rsds"

# Climate model and scenario experiment.
model = "CNRM-ESM2-1"
experiment = "ssp370"

# Reference historical period used to compute the model baseline.
historical_experiment = "historical"
historical_start_year = 1985
historical_end_year = 2014

# Future period used to compute the scenario delta.
start_year = 2020
end_year = 2050

# Months to process. Use range(1, 13) to process the full year.
months = range(1, 2)
month = 1
# Input path: present ATLAS output produced by the scaling notebook.
atlas_path = Path(f"../data/atlas_data/{country}/")

# Input paths: downscaled scenario products.
historical_downscaling_path = Path(
    f"../data/downscaled_data/{target}/{country}/{model}/{historical_experiment}/"
)
scenario_downscaling_path = Path(
    f"../data/downscaled_data/{target}/{country}/{model}/{experiment}/"
)

# Output path: future ATLAS data corrected with climate scenario deltas.
output_path = Path(f"../data/atlas_data/{country}/{model}/{experiment}/")
output_path.mkdir(parents=True, exist_ok=True)

# File naming templates.
present_file_template = f"ssrd_integrated_{country}_m{month}.nc"
historical_file_template = f"{target}_downscaled_{country}_m{month}_1985_2014.nc"
scenario_file_template = f"{target}_downscaled_{country}_m{month}_{start_year}_{end_year}.nc"
output_file_template = f"ssrd_corrected_{country}_m{month}_{experiment}_{start_year}_{end_year}.nc"

# Variable names inside NetCDF files.
present_variable_name = "ssrd_integrated"
downscaled_variable_name = f"{target}_downscaled"
delta_variable_name = "delta"
corrected_variable_name = "ssrd_corrected"

## Step 3. Define helper functions

These functions open the required files, compute the scenario delta, and save the corrected dataset.

No user edits are normally required in this section.


In [3]:
def find_required_file(folder: Path, filename: str) -> Path:
    """Return a required file path and raise a clear error if it is missing."""
    file_path = folder / filename
    if not file_path.exists():
        raise FileNotFoundError(
            f"Required file not found:{file_path}"
            "Check the input parameters and the folder structure."
        )
    return file_path


def open_present_atlas(month: int) -> xr.Dataset:
    """Open the present ATLAS dataset for one month."""
    filename = present_file_template.format(country=country, month=month)
    file_path = find_required_file(atlas_path, filename)
    return xr.open_dataset(file_path)


def open_downscaled_historical(month: int) -> xr.Dataset:
    """Open the downscaled historical model dataset for one month."""
    filename = historical_file_template.format(
        target=target,
        country=country,
        month=month,
        start_year=historical_start_year,
        end_year=historical_end_year,
    )
    file_path = find_required_file(historical_downscaling_path, filename)
    return xr.open_dataset(file_path)


def open_downscaled_scenario(month: int) -> xr.Dataset:
    """Open the downscaled future scenario model dataset for one month."""
    filename = scenario_file_template.format(
        target=target,
        country=country,
        month=month,
        start_year=start_year,
        end_year=end_year,
    )
    file_path = find_required_file(scenario_downscaling_path, filename)
    return xr.open_dataset(file_path)


def save_xarray_netcdf_fast(ds: xr.Dataset, file_path: Path) -> None:
    """Save an xarray Dataset as compressed NetCDF."""
    file_path.parent.mkdir(parents=True, exist_ok=True)
    ds = ds.astype("float32")

    encoding = {}
    for var_name in ds.data_vars:
        var = ds[var_name]
        chunksizes = tuple(min(2048, int(var.sizes[dim])) for dim in var.dims)
        encoding[var_name] = {
            "dtype": "float32",
            "zlib": True,
            "complevel": 1,
            "shuffle": True,
            "chunksizes": chunksizes,
        }

    delayed = ds.to_netcdf(
        file_path,
        engine="h5netcdf",
        encoding=encoding,
        compute=False,
        mode="w",
    )

    with ProgressBar():
        delayed.compute(scheduler="single-threaded")


def calculate_future_atlas(month: int) -> xr.Dataset:
    """Create the future ATLAS dataset for one month using the delta method."""
    present_ds = open_present_atlas(month)
    historical_ds = open_downscaled_historical(month)
    scenario_ds = open_downscaled_scenario(month)

    delta = scenario_ds[downscaled_variable_name] - historical_ds[downscaled_variable_name]
    corrected = present_ds[present_variable_name] + delta

    future_ds = xr.Dataset(
        data_vars={
            corrected_variable_name: corrected,
            delta_variable_name: delta,
        }
    )

    future_ds.attrs.update(
        {
            "description": "Future ATLAS solar dataset corrected using climate scenario deltas.",
            "country": country,
            "model": model,
            "experiment": experiment,
            "historical_period": f"{historical_start_year}-{historical_end_year}",
            "future_period": f"{start_year}-{end_year}",
            "method": "future_ATLAS = present_ATLAS + (scenario_downscaled - historical_downscaled)",
        }
    )

    return future_ds


## Step 4. Run the scenario correction

Run this cell to process all selected months and save the corrected future ATLAS datasets.


In [4]:
for month in months:
    future_ds = calculate_future_atlas(month)

    output_file = output_path / output_file_template.format(
        country=country,
        month=month,
        experiment=experiment,
        start_year=start_year,
        end_year=end_year,
    )

    save_xarray_netcdf_fast(future_ds, output_file)
    print(f"Saved: {output_file}")


ERROR 1: PROJ: proj_create_from_database: Open of /home/alessandrom/anaconda3/envs/bias_correction_conda/share/proj failed


[########################################] | 100% Completed | 103.31 ms
Saved: ../data/atlas_data/argentina/CNRM-ESM2-1/ssp370/ssrd_corrected_argentina_m1_ssp370_2020_2050.nc


## Step 5. Optional quick check

Run this cell only if you want to inspect one output file after processing.


In [5]:
# Example: open the first processed month and inspect the dataset structure.
example_month = list(months)[0]
example_file = output_path / output_file_template.format(
    country=country,
    month=example_month,
    experiment=experiment,
    start_year=start_year,
    end_year=end_year,
)

xr.open_dataset(example_file)


<xarray.Dataset> Size: 8GB
Dimensions:         (latitude: 39515, longitude: 23888)
Coordinates:
  * latitude        (latitude) float32 158kB -55.03 -55.03 ... -21.8 -21.8
  * longitude       (longitude) float32 96kB -73.57 -73.57 ... -53.67 -53.67
    spatial_ref     int64 8B ...
Data variables:
    ssrd_corrected  (latitude, longitude) float32 4GB ...
    delta           (latitude, longitude) float32 4GB ...
Attributes:
    description:        Future ATLAS solar dataset corrected using climate sc...
    country:            argentina
    model:              CNRM-ESM2-1
    experiment:         ssp370
    historical_period:  1985-2014
    future_period:      2020-2050
    method:             future_ATLAS = present_ATLAS + (scenario_downscaled -...